# Court Judgment Extraction Pipeline
### Extract structured metadata from Indian Court Judgment `.txt` files → Excel

**Project**: Legal Judgment Data Extraction (Assamese / Multilingual)

**Columns Extracted:**
| # | Column | Description |
|---|--------|-------------|
| 1 | File Name | Source `.txt` file name |
| 2 | Case Type | WPC, CrlA, FA, etc. (from filename) |
| 3 | Case Number | e.g. 1056 |
| 4 | Case Year | e.g. 1999 |
| 5 | Language | As, Hi, En, etc. (from filename) |
| 6 | Court Name | Extracted from judgment text |
| 7 | Judge(s) | Name(s) of judge(s) |
| 8 | Petitioner | Applicant / Appellant name |
| 9 | Respondent | Opposite party name |
| 10 | Date of Judgment | Decision date |
| 11 | Citation | Case citation references |
| 12 | Acts/Articles Mentioned | Constitutional articles, Acts referred |
| 13 | Cases Cited | Precedent cases referred |
| 14 | Advocates (Petitioner) | Lawyer names for petitioner |
| 15 | Advocates (Respondent) | Lawyer names for respondent |
| 16 | Judgment Summary | Brief summary / headnote |
| 17 | Final Order/Direction | Disposition / final direction of court |

## 1. Install & Import Dependencies

In [ ]:
# !pip install openpyxl pandas

import os
import re
import glob
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime

## 2. Configuration — Set Your Folder Path Here

In [ ]:
# ============================================================
# CONFIGURE THESE PATHS
# ============================================================

# Folder containing all judgment .txt files
INPUT_FOLDER = "/mnt/user-data/uploads"  # <-- change to your folder path

# Output Excel file path
OUTPUT_EXCEL = "judgments_extracted.xlsx"

# File pattern (only .txt files matching judgment naming)
FILE_PATTERN = "*.txt"

# List all files found
files = sorted(glob.glob(os.path.join(INPUT_FOLDER, FILE_PATTERN)))
print(f"Found {len(files)} judgment file(s):")
for f in files:
    print(f"  → {os.path.basename(f)}")

## 3. Filename Parser — Extract Case Type, Number, Year, Language

In [ ]:
def parse_filename(filepath):
    """
    Parse judgment filename to extract metadata.
    Expected formats:
      - WPC_1056_1999_As.txt
      - CrlA_123_2005_Hi.txt
      - 1782715086904_WPC_1056_1999_As.txt  (with upload prefix)
    """
    basename = os.path.splitext(os.path.basename(filepath))[0]
    
    # Try pattern: optional_prefix_CASETYPE_NUMBER_YEAR_LANG
    pattern = r'(?:\d+_)?([A-Za-z]+)_(\d+)_(\d{4})_([A-Za-z]{2})$'
    match = re.search(pattern, basename)
    
    if match:
        return {
            'case_type': match.group(1),
            'case_number': match.group(2),
            'case_year': match.group(3),
            'language': match.group(4)
        }
    
    # Fallback: try any reasonable split
    parts = basename.split('_')
    return {
        'case_type': parts[0] if len(parts) > 0 else '',
        'case_number': parts[1] if len(parts) > 1 else '',
        'case_year': parts[2] if len(parts) > 2 else '',
        'language': parts[3] if len(parts) > 3 else ''
    }

# Test with sample file
if files:
    test = parse_filename(files[0])
    print(f"Test parse: {test}")

## 4. Text Content Extractor — Parse Judgment Body

In [ ]:
def read_judgment(filepath):
    """Read the full text of a judgment file."""
    encodings = ['utf-8', 'utf-16', 'latin-1', 'cp1252']
    for enc in encodings:
        try:
            with open(filepath, 'r', encoding=enc) as f:
                return f.read()
        except (UnicodeDecodeError, UnicodeError):
            continue
    return ""


def split_pages(text):
    """Split text by page markers like '--- Page N ---'."""
    pages = re.split(r'---\s*Page\s+\d+\s*---', text)
    return [p.strip() for p in pages if p.strip()]


def get_first_n_pages(text, n=5):
    """Get text from first N pages (where most metadata lives)."""
    pages = split_pages(text)
    return '\n'.join(pages[:n])


def get_last_n_pages(text, n=3):
    """Get text from last N pages (where final order usually is)."""
    pages = split_pages(text)
    return '\n'.join(pages[-n:])

## 5. Metadata Extraction Functions

In [ ]:
def extract_court_name(text):
    """
    Extract court name — works for multilingual judgments.
    Looks for patterns like 'High Court', 'Supreme Court', 
    or Assamese equivalents like 'উচ্চ ন্যায়ালয়'
    """
    header = text[:1000]  # Court name is always near the top
    
    # English patterns
    en_patterns = [
        r'(?:IN\s+THE\s+)?(.*?(?:HIGH\s+COURT|SUPREME\s+COURT|TRIBUNAL).*?)(?:\n|$)',
        r'((?:High|Supreme)\s+Court\s+(?:of|at)\s+[\w\s]+)',
    ]
    
    # Assamese patterns
    as_patterns = [
        r'(.*উচ্চ\s*ন্যায়ালয়.*?)(?:\n)',     # High Court in Assamese
        r'(.*সৰ্বোচ্চ\s*ন্যায়ালয়.*?)(?:\n)',  # Supreme Court in Assamese  
    ]
    
    # Hindi patterns
    hi_patterns = [
        r'(.*उच्च\s*न्यायालय.*?)(?:\n)',
        r'(.*सर्वोच्च\s*न्यायालय.*?)(?:\n)',
    ]
    
    for pattern_list in [en_patterns, as_patterns, hi_patterns]:
        for pattern in pattern_list:
            match = re.search(pattern, header, re.IGNORECASE)
            if match:
                return match.group(1).strip()
    
    # Fallback: return the 2nd or 3rd non-empty line (usually court name)
    lines = [l.strip() for l in header.split('\n') if l.strip()]
    if len(lines) >= 2:
        return lines[1]
    return ''


def extract_judges(text):
    """Extract judge name(s) from judgment header."""
    header = text[:1500]
    
    patterns = [
        # English
        r'(?:HON.*?JUSTICE|JUSTICE|JUDGE|CORAM)[:\s]+(.+?)(?:\n|,\s*(?:AND|&))',
        r'(?:Before|BEFORE)[:\s]+(?:HON.*?)?(.+?)(?:\n)',
        # Assamese
        r'ন্যায়াধীশ\s+(.+?)(?:[,\n\)])',
        r'বিচাৰপতি\s+(.+?)(?:[,\n\)])',
        # Hindi  
        r'न्यायाधीश\s+(.+?)(?:[,\n\)])',
        r'न्यायमूर्ति\s+(.+?)(?:[,\n\)])',
    ]
    
    judges = []
    for pattern in patterns:
        matches = re.findall(pattern, header, re.IGNORECASE)
        judges.extend([m.strip() for m in matches if m.strip()])
    
    return '; '.join(judges) if judges else ''


def extract_parties(text):
    """
    Extract petitioner and respondent names.
    Looks for 'vs', 'v.', 'বনাম' (Assamese), 'बनाम' (Hindi)
    """
    header = text[:2000]
    petitioner = ''
    respondent = ''
    
    # Pattern: Petitioner ... vs/বনাম/बनाम ... Respondent
    vs_patterns = [
        r'(.+?)\s*(?:\.\.\.+|…+)?\s*(?:আবেদনকাৰী|Petitioner|Appellant|अपीलार्थी)',
        r'(.+?)\s+(?:vs?\.?|বনাম|बनाम)\s+(.+?)\s*(?:\.\.\.+|…+)?\s*(?:প্ৰতিবাদী|Respondent|প্রতিবাদী|प्रतिवादी)',
    ]
    
    # Try to find petitioner
    pet_patterns = [
        r'(.+?)\s*[\.\-_]+\s*(?:আবেদনকাৰী|Petitioner|Appellant)',
        r'(.+?)\s*(?:আবেদনকাৰী|Petitioner|Appellant)',
    ]
    for p in pet_patterns:
        m = re.search(p, header)
        if m:
            petitioner = m.group(1).strip().strip('.-_ ')
            break
    
    # Try to find respondent  
    resp_patterns = [
        r'(?:বনাম|বি/নাম|vs?\.?|बनाम)\s*\n*(.+?)\s*[\.\-_]+\s*(?:প্ৰতিবাদী|Respondent|প্রতিবাদী|प्रतिवादी)',
        r'(?:বনাম|vs?\.?|बनाम)\s*\n*(.+?)(?:\n)',
    ]
    for p in resp_patterns:
        m = re.search(p, header)
        if m:
            respondent = m.group(1).strip().strip('.-_ ')
            break
    
    return petitioner, respondent


def extract_judgment_date(text):
    """Extract the date of judgment/decision."""
    header = text[:2000]
    
    patterns = [
        # English dates
        r'(?:Decided|Dated|Date of (?:Judgment|Decision|Order))\s*[:\-–]?\s*(.+?)(?:\n)',
        r'(\d{1,2}[./\-]\d{1,2}[./\-]\d{2,4})',
        r'(\d{1,2}\s+(?:January|February|March|April|May|June|July|August|September|October|November|December)\s*,?\s*\d{4})',
        # Assamese
        r'সিদ্ধান্ত\s*(?:লোৱা\s*হয়)?\s*[:\-–]?\s*(.+?)(?:\n)',
        r'তাৰিখ\s*[:\-–]?\s*(.+?)(?:\n)',
        # Hindi
        r'निर्णय\s*(?:दिनांक)?\s*[:\-–]?\s*(.+?)(?:\n)',
    ]
    
    for pattern in patterns:
        match = re.search(pattern, header, re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return ''


def extract_citation(text):
    """Extract case citation from the first few lines."""
    first_lines = text[:500]
    
    patterns = [
        r'(\(\d{4}\)\s*\d+\s*\w+\s*\d+)',       # (2000) 2 GauLR 356
        r'(AIR\s*\d{4}\s*\w+\s*\d+)',             # AIR 2001 Gau 83
        r'(\d{4}\s*\w+\s*অনলাইন\s*\w+\s*\d+)',   # Assamese online citation
        r'(এছচিচি\s*অনলাইন.*?\d+)',               # SCC Online
        r'(এআই\s*আৰ.*?\d+)',                      # AIR in Assamese
    ]
    
    citations = []
    for p in patterns:
        matches = re.findall(p, first_lines)
        citations.extend(matches)
    
    return ' ; '.join(citations) if citations else first_lines.split('\n')[0] if first_lines else ''


def extract_acts_articles(text):
    """Extract mentioned Acts, Articles, and Sections."""
    patterns = [
        # English
        r'(Article\s+\d+[\w()]*)',
        r'(Section\s+\d+[\w()]*)',
        r'([\w\s]+Act,?\s*\d{4})',
        # Assamese
        r'(অনুচ্ছেদ\s+[\d()\w]+)',      # Article
        r'(ধাৰা\s+[\d()\w]+)',            # Section
        r'(অনুসূচী)',                      # Schedule
        r'(আইন[,\s]+\d{4})',              # Act, year
        r'(নিয়মাৱলী[,\s]+\d{4})',        # Regulation, year
        # Hindi
        r'(अनुच्छेद\s+[\d()]+)',
        r'(धारा\s+[\d()]+)',
    ]
    
    acts = set()
    for p in patterns:
        matches = re.findall(p, text)
        acts.update([m.strip() for m in matches])
    
    return '; '.join(sorted(acts)) if acts else ''


def extract_cases_cited(text):
    """Extract precedent cases referred in the judgment."""
    patterns = [
        # "X vs Y" or "X v. Y" patterns
        r'([A-Z][\w\s\.]+\s+(?:vs?\.?|বনাম|बनाम)\s+[A-Z][\w\s\.]+)',
        # Assamese case references with AIR/SCC
        r'([\w\s\.]+বনাম[\w\s\.]+)',
        # Citation patterns like (1997) 5 SCC 201
        r'(\(\d{4}\)\s*\d+\s*(?:SCC|SCR|AIR)\s*\d+)',
        r'(AIR\s*\d{4}\s*SC\s*\d+)',
        r'(এআইআৰ\s*\d{4}\s*এছ[\s]*চি\s*\d+)',
    ]
    
    # Look in the "cases cited" section or throughout
    cited_section = ''
    section_markers = [
        r'(?:উল্লেখ\s*কৰা\s*গোচৰ|Cases?\s*(?:cited|referred)|संदर्भित\s*मामले)(.*?)(?:বিচাৰ|JUDGMENT|ORDER|निर्णय)',
    ]
    for marker in section_markers:
        m = re.search(marker, text, re.DOTALL | re.IGNORECASE)
        if m:
            cited_section = m.group(1)
            break
    
    search_text = cited_section if cited_section else text
    
    cases = set()
    for p in patterns:
        matches = re.findall(p, search_text, re.IGNORECASE)
        cases.update([m.strip()[:100] for m in matches if len(m.strip()) > 5])
    
    return '; '.join(sorted(cases)[:10]) if cases else ''  # Limit to 10


def extract_advocates(text):
    """Extract advocate names for petitioner and respondent sides."""
    # Find the advocates section
    adv_section = ''
    section_markers = [
        r'(?:অধিবক্তাসকল|Advocates?|Counsel|अधिवक्ता)[:\s]*(.*?)(?:উল্লেখ|বিচাৰ|JUDGMENT|ORDER|निर्णय)',
        r'(?:হাজিৰ\s*হোৱা\s*অধিবক্তাসকল|Appeared?)[:\s]*(.*?)(?:উল্লেখ|বিচাৰ|JUDGMENT|ORDER|निर्णय)',
    ]
    
    for p in section_markers:
        m = re.search(p, text, re.DOTALL | re.IGNORECASE)
        if m:
            adv_section = m.group(1)
            break
    
    if not adv_section:
        adv_section = text[:5000]
    
    adv_pet = ''
    adv_resp = ''
    
    # Petitioner advocate patterns
    pet_patterns = [
        r'(.+?)\.+\s*আবেদনকাৰীৰ\s*বাবে',
        r'(.+?)\.+\s*(?:for|For)\s*(?:the\s*)?(?:Petitioner|Appellant)',
        r'(.+?)\s+(?:for|For)\s*(?:the\s*)?(?:Petitioner|Appellant)',
        r'(.+?)\s*(?:अपीलार्थी|याचिकाकर्ता)\s*(?:के\s*)?(?:लिए|ओर\s*से)',
    ]
    
    for p in pet_patterns:
        m = re.search(p, adv_section, re.IGNORECASE)
        if m:
            adv_pet = m.group(1).strip().strip('.-_ ,;:\n')
            break
    
    # Respondent advocate patterns  
    resp_patterns = [
        r'আবেদনকাৰীৰ\s*বাবে.*?([\s\S]+?)\.+\s*উত্তৰদাতাসকলৰ\s*বাবে',
        r'আবেদনকাৰীৰ\s*বাবে.*?([\s\S]+?)\.+\s*প্ৰতিবাদী(?:সকলৰ)?\s*বাবে',
        r'(?:for|For)\s*(?:the\s*)?(?:Petitioner|Appellant).*?([\s\S]+?)\s*(?:for|For)\s*(?:the\s*)?(?:Respondent|State)',
        r'(.+?)\s*(?:प्रतिवादी)\s*(?:के\s*)?(?:लिए|ओर\s*से)',
    ]
    
    for p in resp_patterns:
        m = re.search(p, adv_section, re.IGNORECASE | re.DOTALL)
        if m:
            adv_resp = m.group(1).strip().strip('.-_ ,;:\n')
            break
    
    return adv_pet, adv_resp

def extract_summary(text):
    """
    Extract headnote / summary from the judgment.
    Usually appears between the case number and the judgment body,
    as constitutional/legal propositions.
    """
    # Look for headnote section (text between case number and 'JUDGMENT'/'বিচাৰ')
    patterns = [
        r'(?:নং[\-\s]*\d+\s*/\s*\d{4})\s*\n+(.+?)(?:এই\s*গোচৰত\s*হাজিৰ|বিচাৰ\s*আৰু\s*আদেশ)',
        r'(?:No\.?\s*\d+)\s*\n+(.+?)(?:JUDGMENT|ORDER)',
        r'সিদ্ধান্ত\s*লোৱা\s*হয়.*?\n+(.+?)(?:এই\s*গোচৰত|বিচাৰ)',
    ]
    
    for p in patterns:
        m = re.search(p, text, re.DOTALL | re.IGNORECASE)
        if m:
            summary = m.group(1).strip()
            # Clean up and truncate
            summary = re.sub(r'\s+', ' ', summary)
            return summary[:2000]  # Limit length
    
    return ''


def extract_final_order(text):
    """
    Extract the final order/direction from the last part of the judgment.
    """
    last_pages = get_last_n_pages(text, 3)
    
    patterns = [
        # Assamese
        r'(এই\s*লেখ\s*আবেদনখন\s*.*?(?:নিষ্পত্তি|disposed).*?)(?:অনুবাদক|$)',
        r'(গতিকে\s*এই\s*তথ্যসমূহ.*?)(?:অনুবাদক|$)',
        # English
        r'(?:ORDER|DIRECTION|DISPOSED)\s*\n+(.*?)$',
        r'(?:writ petition.*?(?:is|stands)\s+(?:disposed|dismissed|allowed).*?)(?:\n\n|$)',
        # Hindi
        r'(?:आदेश|निर्देश)\s*\n+(.*?)$',
    ]
    
    for p in patterns:
        m = re.search(p, last_pages, re.DOTALL | re.IGNORECASE)
        if m:
            order = m.group(1).strip()
            order = re.sub(r'\s+', ' ', order)
            return order[:2000]
    
    # Fallback: last meaningful paragraph
    paragraphs = [p.strip() for p in last_pages.split('\n\n') if p.strip() and len(p.strip()) > 50]
    if paragraphs:
        return re.sub(r'\s+', ' ', paragraphs[-1])[:2000]
    return ''

## 6. Master Extraction Function — Process One File

In [ ]:
def extract_judgment_data(filepath):
    """
    Master function: extracts all fields from one judgment file.
    Returns a dictionary with all columns.
    """
    filename = os.path.basename(filepath)
    text = read_judgment(filepath)
    
    if not text:
        print(f"  ⚠ Could not read: {filename}")
        return None
    
    # Parse filename metadata
    file_meta = parse_filename(filepath)
    
    # Extract from text content
    petitioner, respondent = extract_parties(text)
    adv_pet, adv_resp = extract_advocates(text)
    
    record = {
        'File Name': filename,
        'Case Type': file_meta.get('case_type', ''),
        'Case Number': file_meta.get('case_number', ''),
        'Case Year': file_meta.get('case_year', ''),
        'Language': file_meta.get('language', ''),
        'Court Name': extract_court_name(text),
        'Judge(s)': extract_judges(text),
        'Petitioner': petitioner,
        'Respondent': respondent,
        'Date of Judgment': extract_judgment_date(text),
        'Citation': extract_citation(text),
        'Acts/Articles Mentioned': extract_acts_articles(text),
        'Cases Cited': extract_cases_cited(text),
        'Advocates (Petitioner)': adv_pet,
        'Advocates (Respondent)': adv_resp,
        'Judgment Summary': extract_summary(text),
        'Final Order/Direction': extract_final_order(text),
    }
    
    return record


# Test on one file
if files:
    test_record = extract_judgment_data(files[0])
    print("\n=== Extracted Fields ===")
    for k, v in test_record.items():
        val_preview = str(v)[:120] + '...' if len(str(v)) > 120 else str(v)
        print(f"  {k}: {val_preview}")

## 7. Batch Processing — Process All Files in Folder

In [ ]:
def process_all_judgments(input_folder, file_pattern="*.txt"):
    """
    Process all judgment files in the folder.
    Returns a list of dictionaries (one per file).
    """
    files = sorted(glob.glob(os.path.join(input_folder, file_pattern)))
    
    if not files:
        print(f"No files found in: {input_folder}")
        return []
    
    print(f"Processing {len(files)} file(s)...\n")
    
    records = []
    for i, fpath in enumerate(files, 1):
        fname = os.path.basename(fpath)
        print(f"[{i}/{len(files)}] {fname}")
        
        try:
            record = extract_judgment_data(fpath)
            if record:
                records.append(record)
                print(f"  ✓ Extracted successfully")
            else:
                print(f"  ✗ Extraction failed")
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nDone! Extracted data from {len(records)}/{len(files)} files.")
    return records


# Run batch processing
all_records = process_all_judgments(INPUT_FOLDER, FILE_PATTERN)

## 8. Create DataFrame & Preview

In [ ]:
df = pd.DataFrame(all_records)
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}\n")
df.head()

## 9. Export to Formatted Excel

In [ ]:
def save_to_excel(df, output_path):
    """
    Save DataFrame to a professionally formatted Excel file.
    """
    wb = Workbook()
    ws = wb.active
    ws.title = "Judgments"
    
    # ---- Styles ----
    header_font = Font(name='Arial', bold=True, color='FFFFFF', size=11)
    header_fill = PatternFill('solid', fgColor='2F5496')
    header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    
    cell_font = Font(name='Arial', size=10)
    cell_alignment = Alignment(vertical='top', wrap_text=True)
    
    thin_border = Border(
        left=Side(style='thin'),
        right=Side(style='thin'),
        top=Side(style='thin'),
        bottom=Side(style='thin')
    )
    
    alt_fill = PatternFill('solid', fgColor='D6E4F0')  # Alternate row color
    
    columns = list(df.columns)
    
    # ---- Write Headers ----
    for col_idx, col_name in enumerate(columns, 1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = thin_border
    
    # ---- Write Data ----
    for row_idx, (_, row) in enumerate(df.iterrows(), 2):
        for col_idx, col_name in enumerate(columns, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=str(row[col_name]) if pd.notna(row[col_name]) else '')
            cell.font = cell_font
            cell.alignment = cell_alignment
            cell.border = thin_border
            if row_idx % 2 == 0:
                cell.fill = alt_fill
    
    # ---- Column Widths ----
    col_widths = {
        'File Name': 30,
        'Case Type': 12,
        'Case Number': 14,
        'Case Year': 12,
        'Language': 10,
        'Court Name': 35,
        'Judge(s)': 30,
        'Petitioner': 30,
        'Respondent': 35,
        'Date of Judgment': 20,
        'Citation': 40,
        'Acts/Articles Mentioned': 45,
        'Cases Cited': 45,
        'Advocates (Petitioner)': 30,
        'Advocates (Respondent)': 30,
        'Judgment Summary': 60,
        'Final Order/Direction': 60,
    }
    
    for col_idx, col_name in enumerate(columns, 1):
        ws.column_dimensions[get_column_letter(col_idx)].width = col_widths.get(col_name, 20)
    
    # ---- Freeze Header Row ----
    ws.freeze_panes = 'A2'
    
    # ---- Auto-filter ----
    ws.auto_filter.ref = ws.dimensions
    
    # ---- Row Height ----
    ws.row_dimensions[1].height = 30
    
    wb.save(output_path)
    print(f"\n✓ Excel saved: {output_path}")
    print(f"  Rows: {len(df)}, Columns: {len(columns)}")


# Save!
save_to_excel(df, OUTPUT_EXCEL)
print(f"\nFile size: {os.path.getsize(OUTPUT_EXCEL) / 1024:.1f} KB")

## 10. Quick Stats & Validation

In [ ]:
# Check extraction quality
print("=== Extraction Quality Report ===")
print(f"Total files processed: {len(df)}\n")

for col in df.columns:
    filled = df[col].apply(lambda x: bool(str(x).strip())).sum()
    pct = (filled / len(df) * 100) if len(df) > 0 else 0
    status = '✓' if pct >= 80 else '⚠' if pct >= 50 else '✗'
    print(f"  {status} {col}: {filled}/{len(df)} ({pct:.0f}%)")

print("\n=== Case Types Distribution ===")
if 'Case Type' in df.columns:
    print(df['Case Type'].value_counts().to_string())

print("\n=== Languages ===")
if 'Language' in df.columns:
    print(df['Language'].value_counts().to_string())

---
## Usage Notes

1. **Set `INPUT_FOLDER`** in Cell 2 to your folder containing `.txt` judgment files  
2. **Filename convention**: `CaseType_CaseNumber_Year_Language.txt` (e.g., `WPC_1056_1999_As.txt`)  
3. **Supported languages**: Assamese (As), Hindi (Hi), English (En) — regex patterns cover all three  
4. **Output**: `judgments_extracted.xlsx` with formatted headers, filters, and alternating row colors  
5. **Add more patterns**: Edit the extraction functions in Cell 5 to handle new formats  
6. **For AI-powered extraction**: Replace regex functions with API calls to Claude/GPT for higher accuracy on complex judgments